# get gene bed

In [23]:
import json, urllib.request
from pathlib import Path

PROJECT_ROOT = Path("../").resolve()
OUT_DIR = PROJECT_ROOT / "tracks" / "ucsc_raw"

GENOME = "hg38"
TRACK = "ncbiRefSeqCurated"   # "ncbiRefSeqSelect" = one transcript per gene
chrom, start1, end1 = "chr19", 11_039_463, 11_163_820   # 1-based inclusive, like the browser

url = (f"https://api.genome.ucsc.edu/getData/track?genome={GENOME};track={TRACK}"
       f";chrom={chrom};start={start1 - 1};end={end1}")
rows = json.load(urllib.request.urlopen(url))[TRACK]

def ints(v):
    """Exon coordinates arrive as '1,2,3,' strings or as lists."""
    return [int(x) for x in (v if isinstance(v, list) else str(v).split(",")) if str(x) != ""]

gene_lines, intron_lines = ['track name="RefSeq genes"'], ['track name="Introns"']
for r in sorted(rows, key=lambda r: (int(r["txStart"]), int(r["txEnd"]))):
    tx_start, tx_end = int(r["txStart"]), int(r["txEnd"])
    starts, ends = ints(r["exonStarts"]), ints(r["exonEnds"])
    strand = r["strand"]
    label = f'{r.get("name2") or r["name"]} {r["name"]}'      # e.g. "LDLR NM_000527.5"

    gene_lines.append("\t".join(map(str, [
        chrom, tx_start, tx_end, label, 0, strand,
        r["cdsStart"], r["cdsEnd"], 0, len(starts),
        ",".join(str(e - s) for s, e in zip(starts, ends)) + ",",
        ",".join(str(s - tx_start) for s in starts) + ",",
    ])))

    n = len(starts) - 1
    for i in range(n):
        number = i + 1 if strand == "+" else n - i              # intron 1 = first in transcription
        intron_lines.append("\t".join(map(str, [
            chrom, ends[i], starts[i + 1], f"{label} intron {number}", 0, strand,
        ])))

OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "genes_refseq.bed").write_text("\n".join(gene_lines) + "\n", newline="\n")
(OUT_DIR / "introns_refseq.bed").write_text("\n".join(intron_lines) + "\n", newline="\n")
print(f"{len(gene_lines) - 1} transcripts, {len(intron_lines) - 1} introns")

28 transcripts, 485 introns


In [24]:
from collections import defaultdict

# Pin a transcript when the automatic choice is wrong: {"SYMBOL": "NM_xxxxxx.x"}
PREFERRED = {
    # "SPC24": "NM_182513.4",
}

def rank(f):
    """Protein-coding first, then longest CDS, then most exons."""
    tx_start, cds_s, cds_e = int(f[1]), int(f[6]), int(f[7])
    sizes = [int(x) for x in f[10].split(",") if x]
    rel = [int(x) for x in f[11].split(",") if x]
    cds_len = sum(max(0, min(tx_start + r + s, cds_e) - max(tx_start + r, cds_s))
                  for r, s in zip(rel, sizes))
    return (f[3].split(" ")[1].startswith("NM_"), cds_len, len(sizes))

# Group transcripts by gene symbol (name field is "SYMBOL ACCESSION")
by_gene = defaultdict(list)
for line in gene_lines[1:]:                      # [0] is the track line
    f = line.split("\t")
    by_gene[f[3].split(" ")[0]].append(f)

url_sel = (f"https://api.genome.ucsc.edu/getData/track?genome=hg38;track=ncbiRefSeqSelect"
           f";chrom={chrom};start={start1 - 1};end={end1}")
# Compare accessions without the version suffix, so NM_000527.5 matches NM_000527.x
canonical = {r["name"].split(".")[0] for r in json.load(urllib.request.urlopen(url_sel))["ncbiRefSeqSelect"]}

def acc(f):
    return f[3].split(" ")[1]

chosen = {}
for symbol, txs in by_gene.items():
    pinned = [f for f in txs if acc(f) == PREFERRED.get(symbol)]
    canon = [f for f in txs if acc(f).split(".")[0] in canonical]
    pick = (pinned or canon or [max(txs, key=rank)])[0]
    how = "pinned" if pinned else "RefSeq Select" if canon else "ranked (no Select transcript)"
    chosen[symbol] = acc(pick)
    print(f"{symbol:12} {chosen[symbol]:16} {how}")

# Keep the chosen transcript; name = symbol, accession moves to column 13
gene_lines_one = [gene_lines[0]]
for line in gene_lines[1:]:
    f = line.split("\t")
    symbol, tx = f[3].split(" ")
    if chosen[symbol] == tx:
        f[3] = symbol
        gene_lines_one.append("\t".join(f + [tx]))

# Introns of the chosen transcripts only; name "SYMBOL ACC intron N" → "SYMBOL intron N"
intron_lines_one = [intron_lines[0]]
for line in intron_lines[1:]:
    f = line.split("\t")
    symbol, tx, _, number = f[3].split(" ")
    if chosen[symbol] == tx:
        f[3] = f"{symbol} intron {number}"
        intron_lines_one.append("\t".join(f))

SMARCA4      NM_003072.5      RefSeq Select
LDLR-AS1     NR_163945.1      ranked (no Select transcript)
LDLR         NM_000527.5      RefSeq Select
MIR6886      NR_106946.1      ranked (no Select transcript)
SPC24        NM_182513.4      RefSeq Select


In [25]:
OUT_DIR = OUT_DIR / ".." / "new_data"
OUT_DIR.mkdir(parents=True, exist_ok=True)

coding = ['track name="Coding genes"']
noncoding = ['track name="Non-coding genes"']
for line in gene_lines_one[1:]:
    f = line.split("\t")
    (noncoding if f[6] == f[7] else coding).append(line)   # f[6], f[7] = thickStart, thickEnd

(OUT_DIR / "genes_coding.bed").write_text("\n".join(coding) + "\n", newline="\n")
(OUT_DIR / "genes_noncoding.bed").write_text("\n".join(noncoding) + "\n", newline="\n")

186